> # ⚠️ Do not run this notebook unattended
>
> The last section, **`# Hardware validation`**, connects to the real board, switches the
> mux's channels, and scans both I2C buses. *Run All* and `jlx run` reach that section.
>
> - **Safe to run on its own:** everything above `# Hardware validation`. It runs on the host
>   against a fake mux, writes the module and its tests, and needs no board.
> - **Hardware section:** run it one cell at a time, reading the **What to do** line above
>   each cell first.

**Where each cell runs.** Everything above `%board connect` runs in ordinary Python (CPython)
on this computer, against `FakeTCA9548A` (written by this notebook to
`devices/muxes/tca9548a_fake.py`) on a `FakeBus`. After `%board connect`, cells run in
MicroPython on the board, except `%%host` cells, which run on this computer. micropylab has
`%%extract` built in, so there is no `%load_ext jupyter_lab_extractor` cell.

# Hardware and configuration
What this notebook was validated with. The same table opens `# Hardware validation`.

| | Host run (simulation) | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13.15 in the micropylab kernel (1.0.0a1), no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0 (v1.28.0, 2026-04-06), USB `/dev/ttyACM0` |
| **I2C bus** | `FakeBus` | both buses scanned to find the mux: bus 0 (STEMMA QT, scl=9 sda=8) and `machine.I2C(1)`, **bus 1**, scl=15 sda=16, 100 kHz (`bus_open("feathers3", 1)`); the checks run on bus 1 |
| **Mux** | `FakeTCA9548A` at `0x71` by default (tests also build it at other addresses) | SparkFun Qwiic Mux (TCA9548A) at `0x70` (address jumpers open) |
| **Behind the mux** | `FakeRegisterPart` at `0x70` on channel 0 (an SHTC3-like part at the mux's own address) and one at `0x48` on channel 2 | channel 0: SparkFun TMAG5273 (B1) at `0x22`; channel 2: TMP102 at `0x48`; channels 1, 3-7 empty (the `BENCH` cell) |
| **Other parts used** | none | seen by the scan, not used: VL53L4CD (`0x29`) and Qwiic Single Relay (`0x18`) on bus 0 |
| **Datasheet** | TI TCA9548A (no copy in `docs/datasheets/`) | same |
| **Result** | 11 unit tests pass (20 cases): 3 for the fake, 8 for the driver | 23 of 23 hardware checks passed (2026-11-13) |

The simulation and the bench differ on purpose: the fake mux sits at `0x71` with a part at `0x70`
behind it, to test the clash a part at the mux's address causes; on the bench the mux is at `0x70`
and nothing behind it shares that address. To rerun the hardware section, wire the bench this way
again (README "The bench").

# TODO:
Nothing

Decided (2026-11-13): the hardware section starts by **finding** the mux (both buses, every
TCA9548A address) instead of assuming bus 0 / `0x71`, because the first TMP102 bring-up got
`ENODEV` at `0x71`.

# Python file
location: `src/micropython_i2c/devices/muxes/tca9548a.py`

fake part: `src/micropython_i2c/devices/muxes/tca9548a_fake.py`

tests: `tests/test_tca9548a.py`

## header

In [1]:
%%extract ../../src/micropython_i2c/devices/muxes/tca9548a.py
"""TI TCA9548A 8-channel I2C multiplexer (the SparkFun Qwiic Mux).

Usage on the board::

    from micropython_i2c.devices.muxes.tca9548a import TCA9548A

    mux = TCA9548A(bus, 0x70, lock=bus_lock)
    mux.channel_select(2)            # only channel 2 connected
    mux.channels_get()               # -> [2]
    mux.channel_scan()               # -> {0: [0x70], 1: [], 2: [0x48], ...}
    mux.channel_deselect()           # everything downstream isolated

One 8-bit control register with no register pointer: each bit enables one channel, so
writing ``1 << ch`` selects exactly that channel and ``0x00`` isolates everything. Power-on
value is ``0x00``. The address is ``0x70`` plus the A2..A0 straps (``0x70``..``0x77``); on
the SparkFun board those are solder jumpers, open by default (``0x70``).

Generated by ``device_notebooks/muxes/tca9548a_devNB``: edit the notebook, not this file.
"""

'TI TCA9548A 8-channel I2C multiplexer (the SparkFun Qwiic Mux).\n\nUsage on the board::\n\n    from micropython_i2c.drivers.tca9548a import TCA9548A\n\n    mux = TCA9548A(bus, 0x70, lock=bus_lock)\n    mux.channel_select(2)            # only channel 2 connected\n    mux.channels_get()               # -> [2]\n    mux.channel_scan()               # -> {0: [0x70], 1: [], 2: [0x48], ...}\n    mux.channel_deselect()           # everything downstream isolated\n\nOne 8-bit control register with no register pointer: each bit enables one channel, so\nwriting ``1 << ch`` selects exactly that channel and ``0x00`` isolates everything. Power-on\nvalue is ``0x00``. The address is ``0x70`` plus the A2..A0 straps (``0x70``..``0x77``); on\nthe SparkFun board those are solder jumpers, open by default (``0x70``).\n\nGenerated by ``dev_notebooks/drivers_tca9548a_devNB``: edit the notebook, not this file.\n'

# Imports
## External
None: the driver runs on MicroPython and uses only this package.

## Internal
`I2CMux` gives the transport and the channel API; `path_select` / `path_release` let
`channel_scan` hold the bus while it switches channels and scans.

In [2]:
%%extract ../../src/micropython_i2c/devices/muxes/tca9548a.py -a
from micropython_i2c.connection import path_release, path_select
from micropython_i2c.node import I2CMux

# Setup

Notebook-only: run the test cells in the notebook. `raise_on_error=True` makes a failing test
fail the run.

In [3]:
import ipytest

ipytest.autoconfig(raise_on_error=True)

# Test file
The test file header: imports only. It is written, not run.

In [4]:
%%writefile ../../tests/test_tca9548a.py
import pytest

from micropython_i2c import NullLock
from micropython_i2c.devices.muxes.tca9548a import TCA9548A
from micropython_i2c.devices.muxes.tca9548a_fake import FakeTCA9548A
from micropython_i2c.sim.bus import FakeBus, FakeMux, FakeRegisterPart

Overwriting ../tests/test_tca9548a.py


Notebook-only: the imports the validation and test cells use. The classes under test (the
driver and the fake) are the ones defined in this notebook, not the modules'.

In [5]:
import pytest

from micropython_i2c import NullLock
from micropython_i2c.sim.bus import FakeBus, FakeMux, FakeRegisterPart

# Intent
**Purpose:** drive a TCA9548A: connect exactly one channel (or a chosen set) to the upstream
bus, isolate them all, read back which are on, and find what sits behind each channel.

**Why it exists:** the routing layer (`I2CConnection`, `RoutingTable`, `I2CManager`) needs a
mux object that implements the `I2CMux` channel API, and bring-up needs a quick answer to
"what is behind channel n?".

**Used by:** `I2CManager` builds one per mux in the routing table (`"driver": "TCA9548A"`).
Connections do **not** call it: they write the one-hot channel byte themselves on every
transfer (`connection.path_select`), under the same bus lock.

**Not its job:** choosing which channel a device needs (the routing table does that).

```mermaid
flowchart LR
    host["board I2C"] --- mux["TCA9548A @ 0x70..0x77"]
    mux --- c0["ch 0"] --- d0["SHTC3 @ 0x70"]
    mux --- c2["ch 2"] --- d2["TMP102 @ 0x48"]
    mux --- cn["ch 1, 3..7"]
```

| Control byte | Meaning |
|---|---|
| `0x00` | every channel off (power-on) |
| `1 << ch` | only channel `ch` |
| any mask | every channel whose bit is set (their devices share the bus: addresses must differ) |

**Address clash.** When a channel is on, the parts behind it and the mux share one bus. A part
with the mux's own address (an SHTC3 is always `0x70`, the mux's default) cannot be told apart
from the mux, so the mux must be strapped elsewhere (`0x71`, A0 jumper closed).

## What the module defines

| Name | What it is |
|---|---|
| `TCA9548A` | the driver: `channel_select`, `channel_mask_set`, `channel_deselect`, `control_read`, `channels_get`, `channel_scan` |

# Fake part
`FakeTCA9548A`: the mux the host run talks to, written to its own module next to the driver so
the bench simulation (`sim.bench`) can use it too. It is a `FakeMux` (the generic one-byte,
one-bit-per-channel mux in `sim.bus`) plus what the TCA9548A datasheet says beyond that. Its
tests go to the same test file as the driver's. It comes before `# Code` because the driver's
validation needs it.

## header

In [ ]:
%%extract ../../src/micropython_i2c/devices/muxes/tca9548a_fake.py
"""``FakeTCA9548A``: a TCA9548A control register for host-side tests.

Attach it to a ``FakeBus`` at its address; ``FakeBus`` routes to the parts behind it by its
``state`` (bit ``n`` set: channel ``n`` connected)::

    from micropython_i2c.sim.bus import FakeBus
    from micropython_i2c.devices.muxes.tca9548a_fake import FakeTCA9548A

    bus = FakeBus()
    bus.attach(0x70, FakeTCA9548A())
    bus.attach(0x48, part, path=[(0x70, 2)])   # answers only while channel 2 is on

Host only.

Generated by ``device_notebooks/muxes/tca9548a_devNB``: edit the notebook, not this file.
"""

## Imports
External: none. Internal: `FakeMux`, the generic mux it extends.

In [ ]:
%%extract ../../src/micropython_i2c/devices/muxes/tca9548a_fake.py -a
from micropython_i2c.sim.bus import FakeMux

## Public names

In [ ]:
%%extract ../../src/micropython_i2c/devices/muxes/tca9548a_fake.py -a
__all__ = ["FakeTCA9548A"]

## `FakeTCA9548A`
What it models, and where each behaviour comes from (TI TCA9548A datasheet; there is no copy in
`docs/datasheets/`):

| Behaviour | Source |
|---|---|
| One 8-bit control register, no register pointer: bit `n` connects channel `n` (from `FakeMux`) | datasheet, "Control Register" |
| Power-on value `0x00`: every channel off (`CONTROL_POR`) | datasheet, "Power-On Reset" and the control register's default |
| A read returns the control register; every byte of a longer read repeats it (from `FakeMux`) | datasheet, "Control Register" (read-back) |
| When a write carries several bytes, the last one is kept (from `FakeMux`) | datasheet, "Control Register" |
| A write with no data byte (address only) leaves the register as it is | inferred from the datasheet's "Control Register": the register is written by a data byte, and there is none |

Not modelled: the `RESET` pin (the driver and the SparkFun board's Qwiic wiring do not use it),
the channel switching taking effect at the STOP condition (`FakeBus` has no STOP; every
transfer is complete), and the electrical side (pull-ups per channel, voltage translation).
Its behaviour does not depend on its address, so it takes no `addr` (no `TAKES_ADDR`).

In [ ]:
%%extract ../../src/micropython_i2c/devices/muxes/tca9548a_fake.py -a
class FakeTCA9548A(FakeMux):
    """A TCA9548A: one control byte, bit ``n`` connects channel ``n``, ``0x00`` at power-on.

    Attributes:
        state: The control register.
    """

    CONTROL_POR = 0x00  # every channel off after power-on

    def __init__(self):
        super().__init__()
        self.state = self.CONTROL_POR

    def raw_write(self, data):
        """Keep the last data byte; an address-only write changes nothing."""
        if data:
            super().raw_write(data)

### validation/test
The checks talk to the fake with raw bus calls (`writeto`, `readfrom`, `scan`): the driver is
not defined yet, and the fake must stand on its own.

#### validation: nominal usage
Power on, select channel 2, read the register back, see the part behind channel 2 appear.

In [ ]:
bus = FakeBus()
bus.attach(0x70, FakeTCA9548A())
bus.attach(0x48, FakeRegisterPart(), path=[(0x70, 2)])
print("power-on control: {:#04x} | bus sees {}".format(
    bus.readfrom(0x70, 1)[0], [hex(a) for a in bus.scan()]))  # fmt: skip
bus.writeto(0x70, bytes([1 << 2]))
print("after selecting channel 2: {:#04x} | bus sees {}".format(
    bus.readfrom(0x70, 1)[0], [hex(a) for a in bus.scan()]))  # fmt: skip

#### validation: anti-nominal usage
The fake has no failure path of its own: every byte value is a valid channel mask, and it
answers at any address it is attached at. A part behind a channel that is off is what fails,
and `FakeBus` raises the `OSError` (`ENODEV`) a real NACK gives.

In [ ]:
bus = FakeBus()
bus.attach(0x70, FakeTCA9548A())
bus.attach(0x48, FakeRegisterPart(), path=[(0x70, 2)])
try:
    bus.readfrom(0x48, 1)  # channel 2 is off at power-on
except OSError as err:
    print("raised OSError, as expected:\n    {}".format(err))
else:
    raise AssertionError("a part behind an off channel should not have answered")

#### unit

Power-on: the control register reads `0x00`, and it is a `FakeMux` (so `FakeBus` routes by
it).

In [ ]:
%%extract ../../tests/test_tca9548a.py -a --strip-ipytest
%%ipytest


def test_fake_powers_on_with_every_channel_off():
    bus = FakeBus()
    mux = bus.attach(0x70, FakeTCA9548A())
    assert isinstance(mux, FakeMux)
    assert mux.state == FakeTCA9548A.CONTROL_POR == 0x00
    assert bus.readfrom(0x70, 1) == b"\x00"

A read returns the control register, every byte of it; a multi-byte write keeps the last
byte.

In [ ]:
%%extract ../../tests/test_tca9548a.py -a --strip-ipytest
%%ipytest


def test_fake_reads_back_the_last_byte_written():
    bus = FakeBus()
    bus.attach(0x70, FakeTCA9548A())
    bus.writeto(0x70, b"\x01\x80\x24")
    assert bus.readfrom(0x70, 3) == b"\x24\x24\x24"

An address-only write (no data byte) leaves the register alone.

In [ ]:
%%extract ../../tests/test_tca9548a.py -a --strip-ipytest
%%ipytest


def test_fake_address_only_write_changes_nothing():
    bus = FakeBus()
    bus.attach(0x70, FakeTCA9548A())
    bus.writeto(0x70, b"\x04")
    bus.writeto(0x70, b"")
    assert bus.readfrom(0x70, 1) == b"\x04"

# Code
## TODO:
Nothing

## Public names

In [6]:
%%extract ../../src/micropython_i2c/devices/muxes/tca9548a.py -a
__all__ = ["TCA9548A"]

## `TCA9548A`
The channel methods write the control byte through the mux's own connection (its upstream
path, so a mux behind another mux works). `channel_scan` is the one method that holds the bus
lock itself: it must switch channels and scan as one sequence, so no other transfer can slip
in between, and it always leaves every channel off.

In [7]:
%%extract ../../src/micropython_i2c/devices/muxes/tca9548a.py -a
class TCA9548A(I2CMux):
    """TCA9548A driver implementing the ``I2CMux`` channel API."""

    ALLOWED_ADDRS = tuple(range(0x70, 0x78))  # 0x70 + A2..A0 straps
    ADDR_DEFAULT = 0x70  # SparkFun Qwiic Mux with every address jumper open
    CHANNELS = 8
    BAUD_NOMINAL = 400_000
    BAUD_MAX = 400_000
    BAUD_MIN = 10_000
    CONTROL_POR = 0x00  # every channel off after power-on or reset

    def channel_select(self, channel):
        """Enable only ``channel``.

        Raises:
            ValueError: If ``channel`` is not 0..7.
        """
        self.channel_check(channel)
        self.write_raw(bytes([1 << channel]))

    def channel_mask_set(self, mask):
        """Enable every channel whose bit is set in ``mask`` (several at once)."""
        self.write_raw(bytes([mask & 0xFF]))

    def channel_deselect(self):
        """Disable every channel."""
        self.write_raw(b"\x00")

    def control_read(self):
        """The control register: one bit per enabled channel."""
        return self.read_raw(1)[0]

    def channels_get(self):
        """The enabled channels, lowest first, decoded from the control register."""
        mask = self.control_read()
        return [ch for ch in range(self.CHANNELS) if mask & (1 << ch)]

    def channel_scan(self, channels=None):
        """Scan behind each channel; return ``{channel: [addresses]}``.

        Each list holds the addresses that answer with that channel on and did **not** answer
        with every channel off, so the mux itself and anything upstream are left out. A part
        at the mux's own address is therefore invisible (the address clash). Holds the bus
        lock for the whole sequence and always ends with every channel off.

        Args:
            channels: The channels to scan; default all.

        Raises:
            ValueError: If a channel is not 0..7 (before any bus traffic).
        """
        chans = list(range(self.CHANNELS)) if channels is None else list(channels)
        for ch in chans:
            self.channel_check(ch)
        bus, path, found = self.conn.bus, self.conn.path, {}
        with self.conn.lock:
            path_select(bus, path)
            try:
                bus.writeto(self.addr, b"\x00")
                root = set(bus.scan())
                for ch in chans:
                    bus.writeto(self.addr, bytes([1 << ch]))
                    found[ch] = sorted(set(bus.scan()) - root)
            finally:
                bus.writeto(self.addr, b"\x00")
                path_release(bus, path)
        return found

### validation/test
A helper that builds the bench on a fake bus: a mux (default `0x71`), an SHTC3-like part at
`0x70` behind channel 0 and a TMP102-like part at `0x48` behind channel 2. It is shared by the
validation cells and the test file.

In [8]:
%%extract ../../tests/test_tca9548a.py -a
def fake_bench(mux_addr=0x71):
    """``(bus, mux)``: a ``TCA9548A`` at ``mux_addr`` with parts behind channels 0 and 2."""
    bus = FakeBus()
    bus.attach(mux_addr, FakeTCA9548A())
    bus.attach(0x70, FakeRegisterPart(), path=[(mux_addr, 0)])
    bus.attach(0x48, FakeRegisterPart(), path=[(mux_addr, 2)])
    return bus, TCA9548A(bus, mux_addr, lock=NullLock())

#### validation: nominal usage
Select a channel, read the control register back, find what is behind each channel.

In [9]:
bus, mux = fake_bench()
mux.channel_select(2)
print("channel 2 on -> control", hex(mux.control_read()), "| channels", mux.channels_get())
print("bus sees:", [hex(a) for a in bus.scan()])
mux.channel_mask_set(0b101)
print("mask 0b101 -> channels", mux.channels_get())
print("channel scan:", {ch: [hex(a) for a in addrs] for ch, addrs in mux.channel_scan().items()})
print("after the scan, channels on:", mux.channels_get())

channel 2 on -> control 0x4 | channels [2]
bus sees: ['0x48', '0x71']
mask 0b101 -> channels [0, 2]
channel scan: {0: ['0x70'], 1: [], 2: ['0x48'], 3: [], 4: [], 5: [], 6: [], 7: []}
after the scan, channels on: []


#### validation: anti-nominal usage
Channel 8 does not exist on an 8-channel mux. The refusal comes before any bus traffic.

In [10]:
bus, mux = fake_bench()
before = len(bus.log)
try:
    mux.channel_select(8)  # channels are 0..7
except ValueError as err:
    print(f"raised ValueError, as expected:\n    {err}")
else:
    raise AssertionError("channel_select(8) should have raised ValueError")
finally:
    print("no bus traffic:", len(bus.log) == before)

raised ValueError, as expected:
    channel 8 out of range 0..7
no bus traffic: True


A bad channel in a scan list is refused before the scan touches the bus.

In [11]:
bus, mux = fake_bench()
before = len(bus.log)
try:
    mux.channel_scan([0, 9])  # 9 is not a channel
except ValueError as err:
    print(f"raised ValueError, as expected:\n    {err}")
else:
    raise AssertionError("channel_scan([0, 9]) should have raised ValueError")
finally:
    print("no bus traffic:", len(bus.log) == before)

raised ValueError, as expected:
    channel 9 out of range 0..7
no bus traffic: True


With the mux at its default `0x70`, the SHTC3-like part behind channel 0 has the same address.
The scan cannot see it: this is the clash the jumper avoids.

In [12]:
bus, mux = fake_bench(mux_addr=0x70)
print(
    "mux at 0x70, channel 0 scan:",
    [hex(a) for a in mux.channel_scan([0])[0]],
    "(the 0x70 part is hidden)",
)

mux at 0x70, channel 0 scan: [] (the 0x70 part is hidden)


#### unit

Selecting a channel writes its one-hot byte, and the control register reads it back.

In [13]:
%%extract ../../tests/test_tca9548a.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize("ch", range(8))
def test_channel_select_enables_only_that_channel(ch):
    _, mux = fake_bench()
    mux.channel_select(ch)
    assert mux.control_read() == 1 << ch
    assert mux.channels_get() == [ch]

........                                                                                     [100%]
========================================= warnings summary =========================================
../.venv/lib/python3.13/site-packages/anyio/pytest_plugin.py:14
  /home/nonyourbusiness/Insync/string12oil@gmail.com/Google Drive/Project_restart/micropython-i2c-toolkit/.venv/lib/python3.13/site-packages/anyio/pytest_plugin.py:14: PytestRemovedIn10Warning: _pytest.python.CallSpec2 has been renamed to CallSpec.
  The CallSpec2 alias will be removed in pytest 10.
  Update imports to use CallSpec instead.
  See https://docs.pytest.org/en/stable/deprecations.html#callspec2-renamed
    from _pytest.python import CallSpec2

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
8 passed, 1 warning in 0.08s


Channels outside 0..7 are refused before any bus traffic.

In [14]:
%%extract ../../tests/test_tca9548a.py -a --strip-ipytest
%%ipytest


@pytest.mark.parametrize("ch", [-1, 8, 99])
def test_out_of_range_channel_is_refused(ch):
    bus, mux = fake_bench()
    before = len(bus.log)
    with pytest.raises(ValueError, match="out of range 0..7"):
        mux.channel_select(ch)
    assert len(bus.log) == before

...                                                                                          [100%]
3 passed in 0.04s


A mask enables several channels at once, and deselect turns them all off.

In [15]:
%%extract ../../tests/test_tca9548a.py -a --strip-ipytest
%%ipytest


def test_mask_set_and_deselect():
    _, mux = fake_bench()
    mux.channel_mask_set(0b1010_0101)
    assert mux.channels_get() == [0, 2, 5, 7]
    mux.channel_deselect()
    assert mux.control_read() == mux.CONTROL_POR
    assert mux.channels_get() == []

.                                                                                            [100%]
1 passed in 0.03s


The parts behind a channel are only on the bus while that channel is selected.

In [16]:
%%extract ../../tests/test_tca9548a.py -a --strip-ipytest
%%ipytest


def test_parts_appear_only_on_their_channel():
    bus, mux = fake_bench()
    mux.channel_deselect()
    assert bus.scan() == [0x71]
    mux.channel_select(2)
    assert bus.scan() == [0x48, 0x71]
    mux.channel_select(0)
    assert bus.scan() == [0x70, 0x71]

.                                                                                            [100%]
1 passed in 0.05s


The channel scan finds each part on its channel, leaves out the mux itself, and ends with
every channel off.

In [17]:
%%extract ../../tests/test_tca9548a.py -a --strip-ipytest
%%ipytest


def test_channel_scan_finds_the_bench():
    _, mux = fake_bench()
    mux.channel_select(5)  # left on before the scan
    found = mux.channel_scan()
    assert found == {0: [0x70], 1: [], 2: [0x48], 3: [], 4: [], 5: [], 6: [], 7: []}
    assert mux.channels_get() == []

.                                                                                            [100%]
1 passed in 0.04s


A part at the mux's own address is invisible to the scan (the documented clash).

In [18]:
%%extract ../../tests/test_tca9548a.py -a --strip-ipytest
%%ipytest


def test_channel_scan_cannot_see_a_part_at_the_mux_address():
    _, mux = fake_bench(mux_addr=0x70)
    assert mux.channel_scan([0, 2]) == {0: [], 2: [0x48]}

.                                                                                            [100%]
1 passed in 0.05s


A bad channel in a scan list is refused before any bus traffic.

In [19]:
%%extract ../../tests/test_tca9548a.py -a --strip-ipytest
%%ipytest


def test_channel_scan_refuses_bad_channels_first():
    bus, mux = fake_bench()
    before = len(bus.log)
    with pytest.raises(ValueError, match="out of range"):
        mux.channel_scan([0, 9])
    assert len(bus.log) == before

.                                                                                            [100%]
1 passed in 0.04s


The datasheet facts: eight channels, addresses `0x70`..`0x77` only, 400 kHz ceiling.

In [20]:
%%extract ../../tests/test_tca9548a.py -a --strip-ipytest
%%ipytest


def test_datasheet_facts():
    assert TCA9548A.CHANNELS == 8
    assert TCA9548A.ALLOWED_ADDRS == tuple(range(0x70, 0x78))
    assert TCA9548A.addr_ok(0x70) and TCA9548A.addr_ok(0x77) and not TCA9548A.addr_ok(0x78)
    assert TCA9548A.baud_ok(400_000) and not TCA9548A.baud_ok(1_000_000)

.                                                                                            [100%]
1 passed in 0.03s


### validation/test: in simulated time
Skipped: nothing about the TCA9548A depends on time. Switching a channel takes effect at the
end of the write (the STOP), there is no conversion, timeout or settling time, and the driver
never waits (`TCA9548A` calls no `sleep_*` or `ticks_*`). A `SimClock` run would only repeat
the routing tests above at different clock values. The time-dependent part of routing is the
device behind the mux, and each device notebook's own simulated-time scenario covers it.

# Hardware validation

> ## ⚠️ STOP — do not run this section unattended
>
> **Everything below talks to the real board.** It connects to the FeatherS3, uploads the
> package, scans both I2C buses and switches the mux's channels.
>
> - Do **not** use *Run All*, *Run All Below* or `jlx run` on this section and walk away.
> - Run it **one cell at a time**. Read the **What to do** line above each cell first.
> - Every cell leaves all mux channels off when it finishes, even if it fails.
> - The section ends with `%board disconnect`. If you stop partway, run that cell yourself.

The section first **finds** the mux on either bus and at any TCA9548A address, then runs a
matching test for every unit test above. What is wired behind each channel is written down
once, in the `BENCH` cell; every check reads it from there.

**Hardware and configuration** (same as the table at the top):

| | Host run (simulation) | Hardware run |
|---|---|---|
| **Runs on** | CPython 3.13.15 in the micropylab kernel (1.0.0a1), no board | Unexpected Maker FeatherS3 (ESP32-S3), MicroPython 1.28.0 (v1.28.0, 2026-04-06), USB `/dev/ttyACM0` |
| **I2C bus** | `FakeBus` | both buses scanned to find the mux: bus 0 (STEMMA QT, scl=9 sda=8) and `machine.I2C(1)`, **bus 1**, scl=15 sda=16, 100 kHz (`bus_open("feathers3", 1)`); the checks run on bus 1 |
| **Mux** | `FakeTCA9548A` at `0x71` by default (tests also build it at other addresses) | SparkFun Qwiic Mux (TCA9548A) at `0x70` (address jumpers open) |
| **Behind the mux** | `FakeRegisterPart` at `0x70` on channel 0 (an SHTC3-like part at the mux's own address) and one at `0x48` on channel 2 | channel 0: SparkFun TMAG5273 (B1) at `0x22`; channel 2: TMP102 at `0x48`; channels 1, 3-7 empty (the `BENCH` cell) |
| **Other parts used** | none | seen by the scan, not used: VL53L4CD (`0x29`) and Qwiic Single Relay (`0x18`) on bus 0 |
| **Datasheet** | TI TCA9548A (no copy in `docs/datasheets/`) | same |
| **Result** | 11 unit tests pass (20 cases): 3 for the fake, 8 for the driver | 23 of 23 hardware checks passed (2026-11-13) |

The simulation and the bench differ on purpose: the fake mux sits at `0x71` with a part at `0x70`
behind it, to test the clash a part at the mux's address causes; on the bench the mux is at `0x70`
and nothing behind it shares that address. To rerun the hardware section, wire the bench this way
again (README "The bench").

## Connect and set up

Notebook-only: connect to the board. Only one notebook can own the board: if the TMP102
notebook still has it, run `%board disconnect` there first.

**What to do:** make sure the board is plugged in and no other notebook owns it.

In [21]:
%board connect

Connected /dev/ttyACM0: FeatherS3 with ESP32-S3 - micropython 1.28.0 (v1.28.0 on 2026-04-06)


Notebook-only: remove the package an earlier upload left on the board, and make sure `/lib`
exists, so the copy below lands exactly at `/lib/micropython_i2c`.

**What to do:** nothing.

In [22]:
import os


def rmtree(path):
    """Delete a board folder and everything in it."""
    for entry in os.ilistdir(path):
        child = path + "/" + entry[0]
        if entry[1] == 0x4000:  # folder
            rmtree(child)
        else:
            os.remove(child)
    os.rmdir(path)


try:
    rmtree("/lib/micropython_i2c")
    print("removed the old /lib/micropython_i2c")
except OSError:
    print("no /lib/micropython_i2c on the board yet")
try:
    os.mkdir("/lib")
except OSError:
    pass  # already there

removed the old /lib/micropython_i2c


Notebook-only: copy `src/micropython_i2c`, including the file this notebook just wrote, to
`/lib/micropython_i2c`. The copy runs on the host; the path is relative to this notebook's folder.

**What to do:** nothing. Takes a few seconds.

In [23]:
%%host
from pathlib import Path

pkg = next(p for p in (Path("../../src"), Path("src")) if p.is_dir()) / "micropython_i2c"
%files copy -r {pkg} :/lib/micropython_i2c

copied ../src/micropython_i2c/__init__.py -> :/lib/micropython_i2c/__init__.py
copied ../src/micropython_i2c/bits.py -> :/lib/micropython_i2c/bits.py
copied ../src/micropython_i2c/boards.py -> :/lib/micropython_i2c/boards.py
copied ../src/micropython_i2c/compat.py -> :/lib/micropython_i2c/compat.py
copied ../src/micropython_i2c/connection.py -> :/lib/micropython_i2c/connection.py
copied ../src/micropython_i2c/crc.py -> :/lib/micropython_i2c/crc.py
copied ../src/micropython_i2c/debug.py -> :/lib/micropython_i2c/debug.py
copied ../src/micropython_i2c/drivers/__init__.py -> :/lib/micropython_i2c/drivers/__init__.py
copied ../src/micropython_i2c/drivers/qwiic_relay.py -> :/lib/micropython_i2c/drivers/qwiic_relay.py
copied ../src/micropython_i2c/drivers/shtc3.py -> :/lib/micropython_i2c/drivers/shtc3.py
copied ../src/micropython_i2c/drivers/tca9548a.py -> :/lib/micropython_i2c/drivers/tca9548a.py
copied ../src/micropython_i2c/drivers/tmag5273.py -> :/lib/micropython_i2c/drivers/tmag5273.py


['../src/micropython_i2c/__init__.py -> :/lib/micropython_i2c/__init__.py',
 '../src/micropython_i2c/bits.py -> :/lib/micropython_i2c/bits.py',
 '../src/micropython_i2c/boards.py -> :/lib/micropython_i2c/boards.py',
 '../src/micropython_i2c/compat.py -> :/lib/micropython_i2c/compat.py',
 '../src/micropython_i2c/connection.py -> :/lib/micropython_i2c/connection.py',
 '../src/micropython_i2c/crc.py -> :/lib/micropython_i2c/crc.py',
 '../src/micropython_i2c/debug.py -> :/lib/micropython_i2c/debug.py',
 '../src/micropython_i2c/drivers/__init__.py -> :/lib/micropython_i2c/drivers/__init__.py',
 '../src/micropython_i2c/drivers/qwiic_relay.py -> :/lib/micropython_i2c/drivers/qwiic_relay.py',
 '../src/micropython_i2c/drivers/shtc3.py -> :/lib/micropython_i2c/drivers/shtc3.py',
 '../src/micropython_i2c/drivers/tca9548a.py -> :/lib/micropython_i2c/drivers/tca9548a.py',
 '../src/micropython_i2c/drivers/tmag5273.py -> :/lib/micropython_i2c/drivers/tmag5273.py',
 '../src/micropython_i2c/drivers/tmp

Notebook-only: drop any modules an earlier run left on the board, import what the checks use
and zero the tally.

**What to do:** nothing.

In [24]:
import sys

for name in [m for m in sys.modules if m.startswith("micropython_i2c")]:
    del sys.modules[name]

from micropython_i2c import default_lock
from micropython_i2c.boards import BOARDS, bus_open
from micropython_i2c.debug import check, check_raises, check_reset, check_summary, info
from micropython_i2c.debug import check_counts  # noqa: F401  (read from the host with %eval)
from micropython_i2c.devices.muxes.tca9548a import TCA9548A

check_reset()

## Find the mux
Scan both buses and look for a TCA9548A address (`0x70`..`0x77`). A bus that cannot be opened
(wrong pins in `boards.py`) is reported, not raised.

**What to do:** nothing. Read the result: it says which bus and address the mux is on.

In [25]:
found = {}
for bus_id, cfg in sorted(BOARDS["feathers3"]["buses"].items()):
    try:
        found[bus_id] = bus_open("feathers3", bus_id, freq=100_000).scan()
        print(
            "bus {} (scl={}, sda={}): {}".format(
                bus_id, cfg["scl"], cfg["sda"], [hex(a) for a in found[bus_id]]
            )
        )
    except Exception as e:  # noqa: BLE001 - a bad pin raises rather than scanning empty
        print(
            "bus {} (scl={}, sda={}): could not open: {}".format(bus_id, cfg["scl"], cfg["sda"], e)
        )

hits = [(b, a) for b, addrs in found.items() for a in addrs if TCA9548A.addr_ok(a)]
if check("a TCA9548A address answers", bool(hits), str([(b, hex(a)) for b, a in hits])):
    MUX_BUS, MUX_ADDR = hits[0]
    if len(hits) > 1:
        info(
            "several 0x70..0x77 answers; using bus {} {:#04x}. An SHTC3 also answers at 0x70.".format(
                MUX_BUS, MUX_ADDR
            )
        )
    print("using the mux at bus {} {:#04x}".format(MUX_BUS, MUX_ADDR))
else:
    info("no mux found: check its Qwiic cable and power LED, and which board port it is on")

bus 0 (scl=9, sda=8): ['0x18', '0x29']
bus 1 (scl=15, sda=16): ['0x70']
  [PASS] a TCA9548A address answers           [(1, '0x70')]
using the mux at bus 1 0x70


Notebook-only: the mux on the bus and address just found, with every channel off.

**What to do:** nothing. If the cell above found no mux, stop here and check the wiring.

In [26]:
bus = bus_open("feathers3", MUX_BUS, freq=100_000)
mux = TCA9548A(bus, MUX_ADDR, lock=default_lock())
mux.channel_deselect()
print("mux at bus {} {:#04x}, channels on: {}".format(MUX_BUS, MUX_ADDR, mux.channels_get()))

mux at bus 1 0x70, channels on: []


Notebook-only: what is wired behind each channel, as `{channel: (part, addresses)}`. A part may
answer at any of its listed addresses (the TMAG5273 board here is at `0x22`; its factory
default is `0x35`). A part at the mux's own address is a **clash**: it cannot be told apart
from the mux, so checks on its channel are reported instead of passed or failed.

**What to do:** edit `BENCH` if the wiring changed, then run the cell.

In [27]:
BENCH = {
    0: ("TMAG5273", (0x22, 0x35)),
    2: ("TMP102", (0x48,)),
}


def clashes(ch):
    """True if a part behind ``ch`` has the mux's own address."""
    return ch in BENCH and MUX_ADDR in BENCH[ch][1]


for ch, (part, addrs) in sorted(BENCH.items()):
    note = "  <- CLASH with the mux" if clashes(ch) else ""
    print("channel {}: {} at {}{}".format(ch, part, [hex(a) for a in addrs], note))

channel 0: TMAG5273 at ['0x22', '0x35']
channel 2: TMP102 at ['0x48']


## Matching tests
One cell per unit test above, with the same name, on the real mux.

**Matches** `test_channel_select_enables_only_that_channel`, for all eight channels.

**What to do:** nothing.

In [28]:
try:
    for ch in range(8):
        mux.channel_select(ch)
        name = "test_channel_select_enables_only_that_channel[{}]".format(ch)
        if clashes(ch):  # two chips answer the control-register read: the value is garbled
            info("{}: {} at the mux address, read-back not checked".format(name, BENCH[ch][0]))
        else:
            check(name, mux.control_read() == 1 << ch and mux.channels_get() == [ch])
finally:
    mux.channel_deselect()

  [PASS] test_channel_select_enables_only_that_channel[0] 
  [PASS] test_channel_select_enables_only_that_channel[1] 
  [PASS] test_channel_select_enables_only_that_channel[2] 
  [PASS] test_channel_select_enables_only_that_channel[3] 
  [PASS] test_channel_select_enables_only_that_channel[4] 
  [PASS] test_channel_select_enables_only_that_channel[5] 
  [PASS] test_channel_select_enables_only_that_channel[6] 
  [PASS] test_channel_select_enables_only_that_channel[7] 


**Matches** `test_out_of_range_channel_is_refused`.

**What to do:** nothing.

In [29]:
for ch in (-1, 8, 99):
    check_raises(
        "test_out_of_range_channel_is_refused[{}]".format(ch), ValueError, mux.channel_select, ch
    )
check("still every channel off", mux.channels_get() == [])

  [PASS] test_out_of_range_channel_is_refused[-1] 
          channel -1 out of range 0..7
  [PASS] test_out_of_range_channel_is_refused[8] 
          channel 8 out of range 0..7
  [PASS] test_out_of_range_channel_is_refused[99] 
          channel 99 out of range 0..7
  [PASS] still every channel off              


True

**Matches** `test_mask_set_and_deselect`.

**What to do:** nothing.

In [30]:
try:
    mux.channel_mask_set(0b1010_0101)
    check("mask 0b10100101 -> channels 0, 2, 5, 7", mux.channels_get() == [0, 2, 5, 7])
    mux.channel_deselect()
    check("test_mask_set_and_deselect", mux.control_read() == TCA9548A.CONTROL_POR)
finally:
    mux.channel_deselect()

  [PASS] mask 0b10100101 -> channels 0, 2, 5, 7 
  [PASS] test_mask_set_and_deselect           


**Matches** `test_parts_appear_only_on_their_channel`: each part in `BENCH` answers only while
its channel is on.

**What to do:** nothing.

In [31]:
try:
    mux.channel_deselect()
    root = bus.scan()
    print("all off:   ", [hex(a) for a in root])
    for ch, (part, addrs) in sorted(BENCH.items()):
        mux.channel_select(ch)
        seen = bus.scan()
        print("channel {}: {}".format(ch, [hex(a) for a in seen]))
        if clashes(ch):
            info("{} on channel {} has the mux address: cannot be told apart".format(part, ch))
        else:
            check("{} only with channel {} on".format(part, ch),
                  any(a in seen and a not in root for a in addrs))  # fmt: skip
finally:
    mux.channel_deselect()

all off:    ['0x70']
channel 0: ['0x22', '0x70']
  [PASS] TMAG5273 only with channel 0 on      
channel 2: ['0x48', '0x70']
  [PASS] TMP102 only with channel 2 on        


**Matches** `test_channel_scan_finds_the_bench`: the whole map in one call, and every channel
off afterwards.

**What to do:** nothing.

In [32]:
mux.channel_select(5)  # left on before the scan, as in the unit test
scan = mux.channel_scan()
for ch, addrs in sorted(scan.items()):
    print("  channel {}: {}".format(ch, [hex(a) for a in addrs]))
for ch, (part, addrs) in sorted(BENCH.items()):
    if clashes(ch):
        info("{} on channel {}: hidden by the clash (expected)".format(part, ch))
    else:
        check(
            "{} on channel {}".format(part, ch),
            any(a in scan[ch] for a in addrs),
            str([hex(a) for a in scan[ch]]),
        )
unexpected = {ch: a for ch, a in scan.items() if a and ch not in BENCH}
if unexpected:
    info(
        "parts on channels not in BENCH: {}".format(
            {ch: [hex(x) for x in a] for ch, a in unexpected.items()}
        )
    )
check("test_channel_scan_finds_the_bench (all off after)", mux.channels_get() == [])

  channel 0: ['0x22']
  channel 1: []
  channel 2: ['0x48']
  channel 3: []
  channel 4: []
  channel 5: []
  channel 6: []
  channel 7: []
  [PASS] TMAG5273 on channel 0                ['0x22']
  [PASS] TMP102 on channel 2                  ['0x48']
  [PASS] test_channel_scan_finds_the_bench (all off after) 


True

**Matches** `test_channel_scan_cannot_see_a_part_at_the_mux_address`. On the real bench this
only applies when a part in `BENCH` has the mux's address; otherwise it is reported and
skipped.

**What to do:** nothing.

In [33]:
clash_chs = [ch for ch in BENCH if clashes(ch)]
if clash_chs:
    hidden = mux.channel_scan(clash_chs)
    check("test_channel_scan_cannot_see_a_part_at_the_mux_address",
          all(MUX_ADDR not in hidden[ch] for ch in clash_chs))  # fmt: skip
else:
    info("no part in BENCH has the mux address {:#04x}: no clash on this bench".format(MUX_ADDR))

  [info] no part in BENCH has the mux address 0x70: no clash on this bench


**Matches** `test_channel_scan_refuses_bad_channels_first`.

**What to do:** nothing.

In [34]:
check_raises("test_channel_scan_refuses_bad_channels_first", ValueError, mux.channel_scan, [0, 9])
check("still every channel off", mux.channels_get() == [])

  [PASS] test_channel_scan_refuses_bad_channels_first 
          channel 9 out of range 0..7
  [PASS] still every channel off              


True

**Matches** `test_datasheet_facts`, on MicroPython.

**What to do:** nothing. No bus traffic.

In [35]:
check("test_datasheet_facts",
      TCA9548A.CHANNELS == 8 and TCA9548A.ALLOWED_ADDRS == tuple(range(0x70, 0x78))
      and TCA9548A.addr_ok(0x77) and not TCA9548A.addr_ok(0x78)
      and TCA9548A.baud_ok(400_000) and not TCA9548A.baud_ok(1_000_000))  # fmt: skip

  [PASS] test_datasheet_facts                 


True

## Tally and disconnect

Notebook-only: the PASS/FAIL tally of every check in this section.

**What to do:** read it. Any `FAIL` line above says which check and why.

In [36]:
check_summary()

23 passed, 0 failed


(23, 0)

Notebook-only: copy the tally to the host before disconnecting, so the last cell can fail the
run after the board is released.

**What to do:** nothing.

In [37]:
%%host
hw_passed, hw_failed = get_ipython().run_line_magic("eval", "check_counts()")

Notebook-only: release the board, so the next run (or another notebook) can have it.

**What to do:** nothing.

In [38]:
%board disconnect

Disconnected /dev/ttyACM0 (port closed, board not reset)


Notebook-only: fail the run if any hardware check failed.

**What to do:** nothing. An error here means a `FAIL` line above.

In [39]:
if hw_failed:
    raise AssertionError("{} of {} hardware checks failed".format(hw_failed, hw_passed + hw_failed))
print("all {} hardware checks passed".format(hw_passed))

all 23 hardware checks passed
